In [32]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("MySparkApp").master("local[*]").getOrCreate())

In [33]:
emp_data = [
    ["001","101","John Doe","30","Male","50000","2015-01-01"],
    ["002","101","Jane Smith","25","Female","45000","2016-02-15"],
    ["003","102","Bob Brown","35","","55000","2014-05-01"],
    ["004","102","Alice Lee","28","Female","48000","2017-09-30"],
    ["005","103","Jack Chan","40","Male","60000","2013-04-01"],
    ["006","103","Jill Wong","32","Female","52000","2018-07-01"],
    ["007","101","James Johnson","42","Male","70000","2012-03-15"],
    ["008","102","Kate Kim","29","Female","51000","2019-10-01"],
    ["009","103","Tom Tan","33","Male","58000","2016-06-01"],
    ["010","104","Lisa Lee","27","Female","47000","2018-08-01"],
    ["011","104","David Park","38","Male","65000","2015-11-01"],
    ["012","105","Susan Chen","31","Female","54000","2017-02-15"],
    ["013","106","Brian Kim","45","Male","75000","2011-07-01"],
    ["014","107","Emily Lee","26","Female","46000","2019-01-01"],
    ["015","106","Michael Lee","37","Male","63000","2014-09-30"],
    ["016","107","Kelly Zhang","30","Female","49000","2018-04-01"],
    ["017","105","George Wang","34","Male","57000","2016-03-15"],
    ["018","104","Nancy Liu","29","Female","50000","2017-06-01"],
    ["019","103","Steven Chen","36","Male","62000","2015-08-01"],
    ["020","102","Grace Kim","32","Female","53000","2018-11-01"]
]

emp_schema = "employee_id string, department_id string, name string, age string, gender string, salary string, hire_date string"


In [34]:
emp = spark.createDataFrame(data=emp_data, schema=emp_schema)

In [35]:
emp.show()

+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|Female| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|  Male| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|Female| 47000|2018-08-01|
|        011|          104|   David Park| 38|  Male| 65000|2015-11-01|
|     

In [36]:
# when case on pyspark

from pyspark.sql.functions import when, col ,expr

emp_gender_fixed = emp.withColumn("gender", when(col("gender") == "Male", "M")
                                    .when(col("gender") == "Female", "F")
                                    .otherwise(None))
emp_gender_fixed.show()

# same using expr() method

emp_gender_fixed_1 = emp.withColumn("new_gender", expr("CASE WHEN gender = 'Male' THEN 'M' WHEN gender = 'Female' THEN 'F' ELSE NULL END"))
emp_gender_fixed_1.show()


+-----------+-------------+-------------+---+------+------+----------+
|employee_id|department_id|         name|age|gender|salary| hire_date|
+-----------+-------------+-------------+---+------+------+----------+
|        001|          101|     John Doe| 30|     M| 50000|2015-01-01|
|        002|          101|   Jane Smith| 25|     F| 45000|2016-02-15|
|        003|          102|    Bob Brown| 35|  null| 55000|2014-05-01|
|        004|          102|    Alice Lee| 28|     F| 48000|2017-09-30|
|        005|          103|    Jack Chan| 40|     M| 60000|2013-04-01|
|        006|          103|    Jill Wong| 32|     F| 52000|2018-07-01|
|        007|          101|James Johnson| 42|     M| 70000|2012-03-15|
|        008|          102|     Kate Kim| 29|     F| 51000|2019-10-01|
|        009|          103|      Tom Tan| 33|     M| 58000|2016-06-01|
|        010|          104|     Lisa Lee| 27|     F| 47000|2018-08-01|
|        011|          104|   David Park| 38|     M| 65000|2015-11-01|
|     

In [37]:
# string replace with regex_replace()

from pyspark.sql.functions import regexp_replace

emp_replaced = emp_gender_fixed_1.withColumn("new_name", regexp_replace(col("name"), "J", "Z"))
emp_replaced.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|      null|    Bob Brown|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill Wong|
|        007|          101|James Johnson| 42|  Male| 70000|2012-03-15|         M|Zames Zohnson|
|        008|          102|     Kate Kim

In [38]:
# date fix using to_date()

from pyspark.sql.functions import to_date

emp_fixed_dates = emp_replaced.withColumn("hire_date", to_date(col("hire_date"), "yyyy-MM-dd"))
emp_fixed_dates.printSchema()

root
 |-- employee_id: string (nullable = true)
 |-- department_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- new_gender: string (nullable = true)
 |-- new_name: string (nullable = true)



In [39]:
# adding current date and timestamp columns

from pyspark.sql.functions import current_date, current_timestamp

emp_with_current_dates = emp_fixed_dates.withColumn("current_date", current_date()).withColumn("current_timestamp", current_timestamp())
emp_with_current_dates.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|  2026-07-26|2026-07-26 07:20:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|  2026-07-26|2026-07-26 07:20:...|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|      null|    Bob Brown|  2026-07-26|2026-07-26 07:20:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-07-26|2026-07-26 07:20:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack 

In [49]:
# how to add timezone

from pyspark.sql.functions import date_format

emp_with_timezone = emp_fixed_dates.withColumn("timezone", date_format(col("current_date"), "z"))
emp_with_timezone.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+--------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|timezone|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+--------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|     UTC|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|     UTC|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|      null|    Bob Brown|     UTC|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|     UTC|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|     UTC|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill Wong|     UTC|
|        007|          101|James Johnson| 42|  Male| 70

In [40]:
# to see all data fully
emp_with_current_dates.show(truncate=False)

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|employee_id|department_id|name         |age|gender|salary|hire_date |new_gender|new_name     |current_date|current_timestamp         |
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------------+
|001        |101          |John Doe     |30 |Male  |50000 |2015-01-01|M         |Zohn Doe     |2026-07-26  |2026-07-26 07:20:40.651971|
|002        |101          |Jane Smith   |25 |Female|45000 |2016-02-15|F         |Zane Smith   |2026-07-26  |2026-07-26 07:20:40.651971|
|003        |102          |Bob Brown    |35 |      |55000 |2014-05-01|null      |Bob Brown    |2026-07-26  |2026-07-26 07:20:40.651971|
|004        |102          |Alice Lee    |28 |Female|48000 |2017-09-30|F         |Alice Lee    |2026-07-26  |2026-07-26 07:20:40.651971|
|005        |103          |Jack Chan    |40 |Mal

In [43]:
# drop null values from a column
emp_dropped_nulls = emp_with_current_dates.na.drop(subset=["new_gender"])
emp_dropped_nulls.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|  2026-07-26|2026-07-26 07:21:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|  2026-07-26|2026-07-26 07:21:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-07-26|2026-07-26 07:21:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack Chan|  2026-07-26|2026-07-26 07:21:...|
|        006|          103|    Jill Wong| 32|Female| 52000|2018-07-01|         F|    Zill 

In [44]:
# instead of dropping null values, we can fill them with a default value using coalesce() function
from pyspark.sql.functions import coalesce, lit
emp_filled_nulls = emp_with_current_dates.withColumn("new_gender", coalesce(col("new_gender"), lit("Unknown")))
emp_filled_nulls.show()

+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|employee_id|department_id|         name|age|gender|salary| hire_date|new_gender|     new_name|current_date|   current_timestamp|
+-----------+-------------+-------------+---+------+------+----------+----------+-------------+------------+--------------------+
|        001|          101|     John Doe| 30|  Male| 50000|2015-01-01|         M|     Zohn Doe|  2026-07-26|2026-07-26 07:27:...|
|        002|          101|   Jane Smith| 25|Female| 45000|2016-02-15|         F|   Zane Smith|  2026-07-26|2026-07-26 07:27:...|
|        003|          102|    Bob Brown| 35|      | 55000|2014-05-01|   Unknown|    Bob Brown|  2026-07-26|2026-07-26 07:27:...|
|        004|          102|    Alice Lee| 28|Female| 48000|2017-09-30|         F|    Alice Lee|  2026-07-26|2026-07-26 07:27:...|
|        005|          103|    Jack Chan| 40|  Male| 60000|2013-04-01|         M|    Zack 

In [45]:
# drop old coloumns and keep only new columns
emp_final = emp_filled_nulls.drop("gender", "name").withColumnRenamed("new_gender", "gender").withColumnRenamed("new_name", "name")
emp_final.show(truncate=False)

+-----------+-------------+---+------+----------+-------+-------------+------------+--------------------------+
|employee_id|department_id|age|salary|hire_date |gender |name         |current_date|current_timestamp         |
+-----------+-------------+---+------+----------+-------+-------------+------------+--------------------------+
|001        |101          |30 |50000 |2015-01-01|M      |Zohn Doe     |2026-07-26  |2026-07-26 07:31:50.771423|
|002        |101          |25 |45000 |2016-02-15|F      |Zane Smith   |2026-07-26  |2026-07-26 07:31:50.771423|
|003        |102          |35 |55000 |2014-05-01|Unknown|Bob Brown    |2026-07-26  |2026-07-26 07:31:50.771423|
|004        |102          |28 |48000 |2017-09-30|F      |Alice Lee    |2026-07-26  |2026-07-26 07:31:50.771423|
|005        |103          |40 |60000 |2013-04-01|M      |Zack Chan    |2026-07-26  |2026-07-26 07:31:50.771423|
|006        |103          |32 |52000 |2018-07-01|F      |Zill Wong    |2026-07-26  |2026-07-26 07:31:50.

In [46]:
# saving as csv

emp_final.write.format("csv").save("data/output/4/emp_final.csv")

In [ ]:
emp_final.write.format("csv").save("data/output/2/emp.csv")